# Modern Portfolio Theory I

Module: Modern Portfolio Theory

## Lesson summary

This notebook introduces mean-variance portfolio construction through return estimates, covariance, simulated weights, efficient frontiers, and Sharpe-ratio comparisons.

## Learning objectives
- Estimate asset returns, volatility, and covariance from market data.
- Simulate portfolios and visualize the risk-return space.
- Identify the minimum variance portfolio and efficient frontier.
- Incorporate the risk-free asset, Sharpe ratio, and tangency portfolio concepts.

## Lesson flow
1. Download and inspect price data.
2. Estimate return and risk inputs.
3. Simulate portfolios and locate efficient allocations.
4. Evaluate risk using both portfolio theory and VaR concepts.


## Setup

In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import date, timedelta


import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

from scipy.stats import norm


sns.set_style("darkgrid")
plt.rc("figure", figsize=(16, 6))

In [ ]:
tickers = ['MAT','DIS','KO', 'NVDA','PFE', "AAPL", "META", "TSLA", "^GSPC","MSFT",]
rf = .06

yesterday = str(date.today() - timedelta(days = 1))
print("Today's date:", yesterday)

## Data

In [ ]:
%%time
start_date = "2018-01-01"
print(f"The number of stock to download are {len(tickers)}")
all_data = yf.download(tickers, start_date, yesterday)
price_data =  all_data["Adj Close"].copy() # pd.DataFrame({ticker: data['Adj Close'] for ticker, data in all_data.items()})
price_data.info()

In [ ]:
price_data.head()

In [ ]:
price_data.info()

In [ ]:
price_data.isna().sum()

In [ ]:
price_data.reset_index().Date.agg([min,max])

In [ ]:
price_data.reset_index().agg([min,max])

In [ ]:
price_data.agg([min,max]).pct_change().dropna()

In [ ]:
price_data.iloc[[0,-1]]

In [ ]:
100000 * (164.309998/21.368668)

In [ ]:
price_data.iloc[[0,-1]].pct_change().dropna().reset_index(drop=True).T.sort_values(
    by=0,
    ascending=False
).plot(
    kind='bar',
    legend=False,
    figsize=(12,7)
);

In [ ]:
price_data.iloc[[0,-1]].pct_change().dropna().reset_index(drop=True).T.sort_values(
    by=0,
    ascending=False
)

In [ ]:
price_data.plot(figsize=(14,9))

In [ ]:
price_data.drop(columns='^GSPC').plot(figsize=(14,9));

In [ ]:
price_data.MAT.plot(figsize=(14,9));

In [ ]:
df_log = np.log(price_data.pct_change().dropna() +1)
df_log.head()

In [ ]:
df_log.hist(
    figsize=(10,10),
    bins=25
);

In [ ]:
df_log.plot(figsize=(18,9), alpha=0.6);

## Estimators

In [ ]:
avg_daily = df_log.mean()
std_daily = df_log.std()
cov_daily = df_log.cov()

avg_ann = avg_daily * 252
std_ann = std_daily * (252**(1/2))
cov_ann = cov_daily * 252

In [ ]:
avg_ann.sort_values()

In [ ]:
avg_ann.sort_values().plot(
    kind='bar'
);

In [ ]:
cov_ann

In [ ]:
fig, ax = plt.subplots(figsize=(14,7))    
sns.heatmap(
    df_log.corr(method='pearson'), 
    vmin=-1, 
    vmax=1,
    annot=True,cmap="rocket_r",
    ax=ax
)
plt.show()

In [ ]:
df_log.var() * 252

In [ ]:
df_parameters_stocks = pd.concat(
    [
        avg_ann,
        (df_log.var() * 252) ** (1/2)
    ],
    axis = 1
)

df_parameters_stocks.columns = ['return', 'volatility']

df_parameters_stocks['sharpe_ratio'] = (df_parameters_stocks['return'] - rf) / df_parameters_stocks.volatility

df_parameters_stocks['sharpe_ratio'].sort_values().plot(kind='barh');


In [ ]:
df_parameters_stocks['return'].sort_values().plot(kind='barh');

In [ ]:
df_parameters_stocks

## Simulated portfolios

In [ ]:
%%time
port_returns = []
port_volatility = []
lst_weights = []

num_assets = len(tickers)
num_portfolios = 50000

for single_portfolio in range(num_portfolios):

  weights = np.array([random.randint(-1000,1000) for x in range(num_assets)], dtype=float)
  weights = weights / np.sum(weights)
  
  returns = np.dot(weights, avg_ann)
  volatility = np.sqrt(np.dot(weights.T, np.dot(cov_ann,weights)))
  if volatility <= .9:
    port_returns.append(returns)
    port_volatility.append(volatility)
    lst_weights.append([round(weight,2) for weight in weights])


df_portfolios = pd.DataFrame(
    {
        'return':port_returns,
        'volatility':port_volatility,
        'weights':lst_weights
    }
)
df_portfolios.head()


In [ ]:
df_portfolios['return'].agg(['mean', 'std'])

In [ ]:
df_portfolios['return'].hist(bins=40);

In [ ]:
df_portfolios.info()

In [ ]:
avg_ann.head()

In [ ]:
avg_ann.loc['MAT']

In [ ]:
std_ann

In [ ]:
fig = plt.figure(figsize=(15,8))
ax1 = fig.add_subplot(111)

ax1.scatter(df_portfolios['volatility'], df_portfolios['return'], s=3, c='r', marker='o', label='simulation', alpha=.3)

for stock in tickers:
  ax1.scatter(std_ann.loc[stock], avg_ann.loc[stock], s=40, marker='x', label=stock)

plt.xlabel('Volatility (Annualized STD)')
plt.ylabel('Expected Return (Annualized AVG)')
plt.title('Simulated portfolios')

plt.legend(loc='upper left')
plt.axvline(.2)
plt.axhline(df_portfolios['return'].max(), color='green')
plt.show();

In [ ]:
df_portfolios.iloc[df_portfolios['return'].idxmax()]

In [ ]:
df_portfolios.iloc[df_portfolios['volatility'].idxmin()]

In [ ]:
tickers

## Minimum variance portfolio (MVP)

In [ ]:
def get_mvp_portfolio(Cov):
  inv_Cov = np.linalg.inv(Cov)
  M_1 = np.zeros((Cov.shape[0],)) + 1 

  num = inv_Cov @ M_1
  den = M_1.T @ inv_Cov @ M_1

  W_mvp = num /den

  return W_mvp

In [ ]:
W_mvp = get_mvp_portfolio(cov_ann)
print(W_mvp)

In [ ]:
return_mvp = W_mvp.T @ avg_ann
volatility_mvp = (W_mvp.T @ cov_ann @ W_mvp)**0.5

print(f"Return mvp portfolio: {return_mvp}")
print(f"Volatility mvp portfolio: {volatility_mvp}")

In [ ]:
fig = plt.figure(figsize=(15,8))
ax1 = fig.add_subplot(111)

ax1.scatter(df_portfolios['volatility'], df_portfolios['return'], s=3, c='r', marker='o', label='simulation', alpha=.3)

for stock in tickers:
  ax1.scatter(std_ann.loc[stock], avg_ann.loc[stock], s=40, marker='x', label=stock)

ax1.scatter(volatility_mvp, return_mvp, s=40, marker='*', label='MVP Portfolio')

plt.xlabel('Volatility (Annualized STD)')
plt.ylabel('Expected Return (Annualized AVG)')
plt.title('Simulated portfolios')

plt.legend(loc='upper left')
plt.axvline(.2)
plt.axhline(df_portfolios['return'].max(), color='green')
plt.show();

## Efficient frontier

In [ ]:
np.zeros((len(cov_ann), )) + 1 

In [ ]:
def get_efficient_frontier(Cov, E, rp):
  '''
  Calculate efficient frontier given the following parameters: 
    - Cov: Covariance matrix
    - E: Vector of estimated returns
    - rp: Expected return of portfolio
  '''
  M_1 = np.zeros((len(Cov), )) + 1 
  inv_Cov = np.linalg.inv(Cov)
  A = (M_1.T @ inv_Cov @ E)
  B = (E.T @ inv_Cov @ E)
  C = (M_1.T @ inv_Cov @ M_1)
  D = B * C - A**2

  g = (1/D) * ((B * inv_Cov @ M_1) - (A * inv_Cov @ E ))
  h = (1/D) * ((C * inv_Cov @ E) - (A * inv_Cov @ M_1 ))

  W_efficient_frontier = g + (h * rp)

  return W_efficient_frontier

In [ ]:
get_efficient_frontier(cov_ann, avg_ann, 0.25)

In [ ]:
W_15_mvp = get_efficient_frontier(cov_ann, avg_ann, 0.25)
return_15_mvp = W_15_mvp.T @ avg_ann
volatility_15_mvp = (W_15_mvp.T @ cov_ann @ W_15_mvp)**0.5

print(f"Return mvp portfolio: {return_15_mvp}")
print(f"Volatility mvp portfolio: {volatility_15_mvp}")

In [ ]:
fig = plt.figure(figsize=(15,8))
ax1 = fig.add_subplot(111)
expected_return = .25

ax1.scatter(df_portfolios['volatility'], df_portfolios['return'], s=3, c='r', marker='o', label='simulation', alpha=.3)

for stock in tickers:
  ax1.scatter(std_ann.loc[stock], avg_ann.loc[stock], s=40, marker='x', label=stock)

ax1.scatter(volatility_mvp, return_mvp, s=40, marker='*', label='MVP Portfolio')
ax1.scatter(volatility_15_mvp, return_15_mvp, s=40, marker='*', label=f'EF {expected_return:.0%}')

plt.xlabel('Volatility (Annualized STD)')
plt.ylabel('Expected Return (Annualized AVG)')
plt.title('Simulated portfolios')

plt.legend(loc='upper left')
plt.axvline(.2)
plt.axhline(df_portfolios['return'].max(), color='green')
plt.axhline(.15, color='yellow')
plt.show();

In [ ]:
n_points = 100
max_return = 1.5
def optimal_weights(n_points, E, cov, r_max):
  w_mvp = get_mvp_portfolio(cov)
  r_min = w_mvp.T @ E
  expected_returns = np.linspace(r_min, r_max, n_points)
  weights = [get_efficient_frontier(cov, E, rp) for rp in expected_returns]
  return weights

weights_ef = optimal_weights(n_points, avg_ann, cov_ann, max_return)
portfolio_returns = []
portfolio_volatility = []
portfolio_weights = []
for w in weights_ef:
  returns = np.dot(w, avg_ann)
  volatility = np.sqrt(np.dot(w.T, np.dot(cov_ann, w)))

  portfolio_returns.append(returns)
  portfolio_volatility.append(volatility)
  portfolio_weights.append([round(x,2) for x in w])

df_ef = pd.DataFrame(
    {
        'return':portfolio_returns,
        'volatility':portfolio_volatility,
        'weights': portfolio_weights
    }
)

df_ef.head()


In [ ]:
fig = plt.figure(figsize=(15,8))
ax1 = fig.add_subplot(111)

ax1.scatter(df_portfolios['volatility'], df_portfolios['return'], s=3, c='r', marker='o', label='simulation', alpha=.3)
ax1.scatter(df_ef['volatility'], df_ef['return'], s=10, c='green', marker='o', label='efficient frontier', alpha=.3)

for stock in tickers:
  ax1.scatter(std_ann.loc[stock], avg_ann.loc[stock], s=40, marker='x', label=stock)

ax1.scatter(volatility_mvp, return_mvp, s=40, marker='*', label='MVP Portfolio')
# ax1.scatter(volatility_15_mvp, return_15_mvp, s=40, marker='*', label='EF 15%')

plt.xlabel('Volatility (Annualized STD)')
plt.ylabel('Expected Return (Annualized AVG)')
plt.title('Simulated portfolios')

plt.legend(loc='upper left')
# plt.axvline(.2)
# plt.axhline(df_portfolios['return'].max(), color='green')
# plt.axhline(.15, color='yellow')
plt.show();

## Efficient portfolio with risk free asset

In [ ]:
def get_efficient_portfolio_with_rf(Cov, E, rp, rf):
  '''
  Calculate the efficient portfolio with risk free asset given the following parameters: 
    - Cov: Covariance matrix
    - E: Vector of estimated returns
    - rp: Expected return of portfolio
    - rf: Risk free asset return
  '''

  M_1 = np.zeros(len(Cov)) + 1
  inv_Cov = np.linalg.inv(Cov)

  H = (E-rf).T @ inv_Cov @ (E-rf)

  W = inv_Cov @ (E-rf)  * (rp-rf)/H
  Wf = 1 - W.sum()

  return W, Wf

In [ ]:
n_points = 100
min_return = .18
max_return = 1.5
def optimal_weights_rf(n_points, E, cov, r_min, r_max,rf):
    expected_returns = np.linspace(r_min, r_max, n_points)
    weights = [get_efficient_portfolio_with_rf(cov, E, rp, rf) for rp in expected_returns]
    return weights

weights_ef_rf = optimal_weights_rf(n_points, avg_ann, cov_ann, min_return, max_return,rf)
portfolio_returns = []
portfolio_volatility = []
portfolio_weights = []
for w, wf in weights_ef_rf:
    returns = np.dot(w, avg_ann) + wf * rf
    volatility = np.sqrt(np.dot(w.T, np.dot(cov_ann, w)))

    portfolio_returns.append(returns)
    portfolio_volatility.append(volatility)
    portfolio_weights.append([round(x,2) for x in w] + [round(wf,2)])

df_ef_rf = pd.DataFrame(
    {
        'return':portfolio_returns,
        'volatility':portfolio_volatility,
        'weights': portfolio_weights
    }
)

df_ef_rf.head()

In [ ]:
fig = plt.figure(figsize=(15,8))
ax1 = fig.add_subplot(111)

ax1.scatter(df_portfolios['volatility'], df_portfolios['return'], s=3, c='r', marker='o', label='Simulation', alpha=.3)
ax1.scatter(df_ef['volatility'], df_ef['return'], s=10, c='green', marker='o', label='Efficient frontier', alpha=.3)
ax1.scatter(df_ef_rf['volatility'], df_ef_rf['return'], s=10, c='blue', marker='o', label='Efficient frontier with risk free asset', alpha=.3)

# for stock in tickers:
#   ax1.scatter(std_ann.loc[stock], avg_ann.loc[stock], s=40, marker='x', label=stock)

ax1.scatter(volatility_mvp, return_mvp, s=40, marker='*', label='MVP Portfolio')


plt.xlabel('Volatility (Annualized STD)')
plt.ylabel('Expected Return (Annualized AVG)')
plt.title('Simulated portfolios')

plt.legend(loc='upper left')
plt.show();

## Sharpe ratio

In [ ]:
df_portfolios.head()

In [ ]:
df_portfolios['sharpe_ratio'] = (df_portfolios['return'] - rf) / df_portfolios.volatility

In [ ]:
df_portfolios['sharpe_ratio'].agg([min,max])

In [ ]:
df_portfolios['return'].hist(bins=25);

In [ ]:
fig = plt.figure(figsize=(15,8))
ax1 = fig.add_subplot(111)

ax1.scatter(df_portfolios['volatility'], df_portfolios['return'], s=3, c=df_portfolios['sharpe_ratio'], marker='o', label='Simulation', alpha=.3)
ax1.scatter(df_ef['volatility'], df_ef['return'], s=10, c='red', marker='o', label='Efficient frontier', alpha=.3)
ax1.scatter(df_ef_rf['volatility'], df_ef_rf['return'], s=10, c='blue', marker='o', label='Efficient frontier with risk free asset', alpha=.3)

ax1.scatter(volatility_mvp, return_mvp, s=40, marker='*', label='MVP Portfolio')

plt.xlabel('Volatility (Annualized STD)')
plt.ylabel('Expected Return (Annualized AVG)')
plt.title('Simulated portfolios')

plt.legend(loc='upper left')
plt.show();

In [ ]:
df_portfolios.sharpe_ratio.hist(bins=50);

## Tangency portfolio

In [ ]:
def get_tangency_portfolio(covariance_matrix, expected_returns, risk_free_rate):
    # Calculate the inverse of the covariance matrix
    inverse_covariance_matrix = np.linalg.inv(covariance_matrix)
    
    # Calculate the vector of excess returns
    excess_returns = expected_returns - risk_free_rate
    
    # Calculate the numerator of the tangency portfolio weights
    numerator = np.dot(inverse_covariance_matrix, excess_returns)
    
    # Calculate the denominator of the tangency portfolio weights
    denominator = np.dot(np.dot(excess_returns, inverse_covariance_matrix), excess_returns)
    
    # Calculate the tangency portfolio weights
    tangency_weights = numerator / denominator
    
    return tangency_weights

In [ ]:
# W_tg = compute_tangency_portfolio(
#     cov_ann, rf,avg_ann
# )

In [ ]:
# def get_tangency_portfolio(Cov, E, rf):
#     inv_Cov = np.linalg.inv(Cov)
#     M_1 = np.zeros((Cov.shape[0],)) + 1 

#     num = inv_Cov @ (E - (rf * (np.zeros((Cov.shape[0],)) + 1) ))
#     den = M_1.T @ inv_Cov @ (E - (rf * (np.zeros((Cov.shape[0],)) + 1) ))

#     W_tg = num /den

#     return W_tg

In [ ]:
W_tg = get_tangency_portfolio(cov_ann, avg_ann, rf)
W_tg

In [ ]:
return_tg = W_tg.T @ avg_ann
volatility_tg = (W_tg.T @ cov_ann @ W_tg)**0.5

print(f"Return tangency portfolio: {return_tg}")
print(f"Volatility tangency portfolio: {volatility_tg}")
print(f"Sharpe ratio: {(return_tg - rf)/volatility_tg}")

In [ ]:
fig = plt.figure(figsize=(15,8))
ax1 = fig.add_subplot(111)

ax1.scatter(df_portfolios['volatility'], df_portfolios['return'], s=3, c=df_portfolios['sharpe_ratio'], marker='o', label='Simulation', alpha=.3)
ax1.scatter(df_ef['volatility'], df_ef['return'], s=10, c='red', marker='o', label='Efficient frontier', alpha=.3)
ax1.scatter(df_ef_rf['volatility'], df_ef_rf['return'], s=10, c='blue', marker='o', label='Efficient frontier with risk free asset', alpha=.3)

ax1.scatter(volatility_mvp, return_mvp, s=80, marker='*', label='MVP Portfolio')
ax1.scatter(volatility_tg, return_tg, s=80, marker='*', label='Tangency Portfolio')

plt.xlabel('Volatility (Annualized STD)')
plt.ylabel('Expected Return (Annualized AVG)')
plt.title('Mean-variance portfolio (MPT)')

plt.legend(loc='upper left')
plt.show();

In [ ]:
df_ef.max()

## Value at Risk (VaR)

### Parametric

In [ ]:
initial_investment = 1000000
conf_level = 0.95

In [ ]:
def VaR_parametric(initial_investment, conf_level):    
    alpha = norm.ppf(1 - conf_level, avg_daily, std_daily) 
    VaR_daily = (initial_investment - initial_investment * (1 + alpha))
    df_var = pd.DataFrame(
        {
            'stock':price_data.columns,
            'alpha':alpha,
            'daily_var':VaR_daily,
        }
    )
    return df_var

In [ ]:
VaR_param = VaR_parametric(initial_investment, conf_level)
VaR_param.sort_values(
    by='daily_var'
).set_index(
    'stock'
).daily_var.plot(
    kind='barh'
);

In [ ]:
VaR_param

In [ ]:
df_var_horizons = pd.concat(
    [
        VaR_param.daily_var*np.sqrt(x) for x in range(1,31)
    ],axis=1
).T

df_var_horizons.columns = VaR_param.stock.to_list()

df_var_horizons = df_var_horizons.reset_index(drop=True)
df_var_horizons.index  = df_var_horizons.index + 1
df_var_horizons.head()

In [ ]:
df_var_horizons.plot();

In [ ]:
def calculate_portfolio_var(portfolio_value, weights, mean_returns, covariance_matrix, confidence_level):
    # Calculate the portfolio return
    portfolio_return = np.dot(weights, mean_returns)
    
    # Calculate the portfolio standard deviation
    portfolio_std_dev = np.sqrt(np.dot(np.dot(weights, covariance_matrix), weights.T))
    
    # Calculate the Z-score corresponding to the confidence level
    z_score = norm.ppf(1 - confidence_level)
    
    # Calculate the VaR
    portfolio_var = portfolio_value * (portfolio_return - z_score * portfolio_std_dev)
    
    return portfolio_var

In [ ]:
calculate_portfolio_var(10000, W_mvp, avg_ann, cov_ann, .95)

## Model limitations

- Mean-variance optimization is highly sensitive to expected-return and covariance estimates.
- The efficient frontier assumes stable inputs, a single-period objective, and frictionless rebalancing.
- Portfolio weights should be interpreted alongside concentration, turnover, and robustness checks.
